# Extract Oil Contract Codes from CFTC Disaggregated Report

Load `disaggregated_combined.csv` and extract all unique contract codes
under commodities related to crude oil, heating oil, gasoline, and gasoil.

In [33]:
import pandas as pd

In [34]:
df = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)
print(f"Total rows: {len(df):,}")
print(f"Unique commodity_name values: {df['commodity_name'].nunique()}")

Total rows: 184,455
Unique commodity_name values: 58


## All Commodity Names in the Report

In [35]:
import csv
from pathlib import Path

all_commodities = (
    df.groupby("commodity_name")["cftc_contract_market_code"]
    .nunique()
    .reset_index()
    .rename(columns={"cftc_contract_market_code": "number_of_contracts"})
    .sort_values("number_of_contracts", ascending=False)
    .reset_index(drop=True)
)

# Label each commodity_name with a future group
name_upper = all_commodities["commodity_name"].str.upper()
all_commodities["future_group"] = ""
all_commodities.loc[name_upper.str.contains("CRUDE", na=False), "future_group"] = "Crude Oil"
all_commodities.loc[name_upper.str.contains("HEATING|DIESEL", regex=True, na=False), "future_group"] = "Heating Oil"
all_commodities.loc[name_upper.str.contains("GASOLINE|UNLEADED", regex=True, na=False), "future_group"] = "Gasoline"
all_commodities.loc[all_commodities["commodity_name"] == "HEATING OIL-DIESEL-GASOIL", "future_group"] = "Heating Oil / Gasoil"

output_path = Path("../docs/commodity_name_to_future_group.csv")
all_commodities.to_csv(output_path, index=False, quoting=csv.QUOTE_NONNUMERIC, encoding="utf-8-sig")
print(f"Written to {output_path.resolve()}")
print(f"Total unique commodity names: {len(all_commodities)}\n")
all_commodities

Written to /Users/oualid/Documents/Projects/omroot_repos/cot-ingest/docs/commodity_name_to_future_group.csv
Total unique commodity names: 58



,commodity_name,number_of_contracts,future_group
0,ELECTRICITY,211,
1,NATURAL GAS,119,
2,POLLUTION,108,
3,CRUDE OIL,36,Crude Oil
4,NATURAL GAS LIQUIDS,34,
5,FUEL OIL,29,
6,HEATING OIL-DIESEL-GASOIL,14,Heating Oil / Gasoil
7,GASOLINE,12,Gasoline
8,COAL,7,
9,FUEL OIL/CRUDE OIL,6,Crude Oil


## Commodity Name to Future Group Mapping

In [ ]:
import csv
from pathlib import Path

# Count unique contracts per commodity_name
commodity_counts = (
    df.groupby("commodity_name")["cftc_contract_market_code"]
    .nunique()
    .reset_index()
    .rename(columns={"cftc_contract_market_code": "number_of_contracts"})
)

# Label each commodity_name with a future group
name_upper = commodity_counts["commodity_name"].str.upper()

commodity_counts["future_group"] = ""
commodity_counts.loc[name_upper.str.contains("CRUDE", na=False), "future_group"] = "Crude Oil"
commodity_counts.loc[name_upper.str.contains("HEATING|DIESEL", regex=True, na=False), "future_group"] = "Heating Oil"
commodity_counts.loc[name_upper.str.contains("GASOLINE|UNLEADED", regex=True, na=False), "future_group"] = "Gasoline"

# Filter to only rows with a label
oil_commodity_map = (
    commodity_counts[commodity_counts["future_group"] != ""]
    .sort_values(["future_group", "commodity_name"])
    .reset_index(drop=True)
)

# HEATING OIL-DIESEL-GASOIL contains both HO and Gasoil contracts — add a Gasoil row
gasoil_row = oil_commodity_map[oil_commodity_map["commodity_name"] == "HEATING OIL-DIESEL-GASOIL"].copy()
gasoil_row["future_group"] = "Gasoil"
oil_commodity_map = pd.concat([oil_commodity_map, gasoil_row], ignore_index=True)
oil_commodity_map = oil_commodity_map.sort_values(["future_group", "commodity_name"]).reset_index(drop=True)

# Save to CSV
output_path = Path("../docs/commodity_name_to_future_group.csv")
oil_commodity_map.to_csv(output_path, index=False, quoting=csv.QUOTE_NONNUMERIC, encoding="utf-8-sig")
print(f"Written to {output_path.resolve()}")
print(f"{len(oil_commodity_map)} commodity names mapped\n")
oil_commodity_map

Written to /Users/oualid/Documents/Projects/omroot_repos/cot-ingest/docs/commodity_name_to_future_group.csv
12 commodity names mapped



,commodity_name,number_of_contracts,future_group
0,CRUDE OIL,36,Crude Oil
1,FUEL OIL/CRUDE OIL,6,Crude Oil
2,NAPHTHA/CRUDE OIL,4,Crude Oil
3,HEATING OIL-DIESEL-GASOIL,14,Gasoil
4,GASOLINE,12,Gasoline
5,UNLEADED GAS/CRUDE OIL SPREADS,2,Gasoline
6,BIODIESEL/HEATING OIL,2,Heating Oil
7,DIESEL/CRUDE OIL,1,Heating Oil
8,DIESEL/HEATING OIL,4,Heating Oil
9,HEATING OIL-DIESEL-GASOIL,14,Heating Oil


## Commodity Names Relevant to Our Futures (CL, CO, HO, XB, QS)

In [ ]:
OIL_KEYWORDS = [
    "CRUDE",
    "HEATING",
    "DIESEL",
    "GASOLINE",
    "UNLEADED",
]

oil_commodities = all_commodities[
    all_commodities.index.str.upper().str.contains("|".join(OIL_KEYWORDS), regex=True, na=False)
]
print(f"Matching commodity names: {len(oil_commodities)}\n")
oil_commodities

AttributeError: Can only use .str accessor with string values!

## Filter to Oil-Related Commodities

We search `commodity_name` for keywords: CRUDE, HEATING, DIESEL, GASOLINE, UNLEADED, GASOIL,
FUEL OIL, NAPHTHA, JET, BIODIESEL.

In [ ]:
OIL_KEYWORDS = [
    "CRUDE",
    "HEATING",
    "DIESEL",
    "GASOLINE",
    "UNLEADED",
    "GASOIL",
    "FUEL OIL",
    "NAPHTHA",
    "JET",
    "BIODIESEL",
]

mask = df["commodity_name"].str.upper().str.contains("|".join(OIL_KEYWORDS), regex=True, na=False)
oil_data = df[mask].copy()
print(f"Oil-related rows: {len(oil_data):,}")
print(f"\ncommodity_name values found:")
print(oil_data["commodity_name"].value_counts())

## All Unique Contract Codes

In [ ]:
contracts = (
    oil_data[["cftc_contract_market_code", "market_and_exchange_names", "commodity_name"]]
    .drop_duplicates()
    .sort_values(["commodity_name", "cftc_contract_market_code", "market_and_exchange_names"])
    .reset_index(drop=True)
)
print(f"Total unique (code, name) pairs: {len(contracts)}")
print(f"Unique codes: {contracts['cftc_contract_market_code'].nunique()}")
contracts

## Crude Oil Contracts

In [ ]:
crude = contracts[contracts["commodity_name"].str.contains("CRUDE", case=False, na=False)]
print(f"Crude oil (code, name) pairs: {len(crude)}  |  Unique codes: {crude['cftc_contract_market_code'].nunique()}")
print(f"\ncommodity_name values:")
print(crude["commodity_name"].value_counts().to_string())
crude

## Heating Oil / Diesel Contracts

In [ ]:
heating = contracts[contracts["commodity_name"].str.contains("HEATING|DIESEL", case=False, regex=True, na=False)]
print(f"Heating oil / diesel (code, name) pairs: {len(heating)}  |  Unique codes: {heating['cftc_contract_market_code'].nunique()}")
print(f"\ncommodity_name values:")
print(heating["commodity_name"].value_counts().to_string())
heating

## Gasoline Contracts

In [ ]:
gasoline = contracts[contracts["commodity_name"].str.contains("GASOLINE|UNLEADED", case=False, regex=True, na=False)]
print(f"Gasoline (code, name) pairs: {len(gasoline)}  |  Unique codes: {gasoline['cftc_contract_market_code'].nunique()}")
print(f"\ncommodity_name values:")
print(gasoline["commodity_name"].value_counts().to_string())
gasoline

## Fuel Oil / Naphtha / Jet Contracts

In [ ]:
other_oil = contracts[contracts["commodity_name"].str.contains("FUEL OIL|NAPHTHA|JET|BIODIESEL", case=False, regex=True, na=False)]
print(f"Fuel oil / naphtha / jet / biodiesel (code, name) pairs: {len(other_oil)}  |  Unique codes: {other_oil['cftc_contract_market_code'].nunique()}")
print(f"\ncommodity_name values:")
print(other_oil["commodity_name"].value_counts().to_string())
other_oil

## Summary — Codes per Commodity Name

In [ ]:
summary = (
    contracts.groupby("commodity_name")
    .agg(
        unique_codes=("cftc_contract_market_code", "nunique"),
        name_variants=("market_and_exchange_names", "count"),
    )
    .sort_values("unique_codes", ascending=False)
)
summary